# AI Productivity Toolkit: Prompt Engineering & AI Productivity (Module 02)
**Codomax Digital Solutions - Internship Project**

This notebook covers every module objective: zero-shot vs few-shot prompting, roles / context / instructions / constraints, structured JSON output, summarization / classification / generation, AI-assisted coding and debugging, prompt evaluation with a testing report, and the reusable **AI Productivity Toolkit**.

Uses a free model (`google/flan-t5-base`), so no API key is needed. Choose **Runtime, Run all**.

## 0. Setup

In [ ]:
!pip install -q transformers torch

The toolkit package is written to disk below (the same files that live in the GitHub repo).

In [ ]:
import os
os.makedirs('ai_productivity_toolkit', exist_ok=True)

In [ ]:
%%writefile ai_productivity_toolkit/__init__.py
"""AI Productivity Toolkit: reusable prompt templates, structured output helpers,
productivity workflows and a prompt-testing harness. Model-agnostic."""
from .backends import CallableLLM, HFBackend, MockLLM
from .evaluation import (
    TestCase, TestResult, compare_prompts, contains_any, equals_label,
    max_words, run_suite, to_markdown, valid_json_with,
)
from .structured import StructuredOutputError, ask_json, extract_fields, extract_json
from .templates import PromptTemplate
from .workflows import (
    CLASSIFY, DEBUG_CODE, EMAIL, EXPLAIN_CODE, RESEARCH, RESUME, SUMMARIZE,
    classify, debug_code, explain_code, improve_resume_bullet, normalize_label,
    research_outline, summarize, write_email,
)

__version__ = "1.0.0"

__all__ = [
    "CallableLLM", "HFBackend", "MockLLM", "PromptTemplate",
    "StructuredOutputError", "ask_json", "extract_fields", "extract_json",
    "TestCase", "TestResult", "compare_prompts", "contains_any", "equals_label",
    "max_words", "run_suite", "to_markdown", "valid_json_with",
    "CLASSIFY", "DEBUG_CODE", "EMAIL", "EXPLAIN_CODE", "RESEARCH", "RESUME", "SUMMARIZE",
    "classify", "debug_code", "explain_code", "improve_resume_bullet", "normalize_label",
    "research_outline", "summarize", "write_email",
]

In [ ]:
%%writefile ai_productivity_toolkit/backends.py
"""Model backends. Anything with `generate(prompt, **kwargs) -> str` can be used."""
from __future__ import annotations

from typing import Callable, List, Optional, Sequence


class HFBackend:
    """Hugging Face text2text backend (instruction-tuned, runs on Colab CPU)."""

    def __init__(self, model_name: str = "google/flan-t5-base", max_new_tokens: int = 128):
        from transformers import pipeline  # imported lazily

        self.name = model_name
        self.max_new_tokens = max_new_tokens
        self._pipe = pipeline("text2text-generation", model=model_name)

    def generate(self, prompt: str, max_new_tokens: Optional[int] = None) -> str:
        out = self._pipe(
            prompt,
            max_new_tokens=max_new_tokens or self.max_new_tokens,
            do_sample=False,
        )
        return out[0]["generated_text"].strip()


class CallableLLM:
    """Wrap any function `fn(prompt) -> str` (e.g. an API client call)."""

    def __init__(self, fn: Callable[[str], str], name: str = "custom"):
        self.fn = fn
        self.name = name

    def generate(self, prompt: str, **_) -> str:
        return self.fn(prompt)


class MockLLM:
    """Fake model for tests. Pass a list of replies (cycled) or a function."""

    def __init__(self, replies: Optional[Sequence[str]] = None,
                 fn: Optional[Callable[[str], str]] = None, name: str = "mock"):
        self.replies = list(replies or [])
        self.fn = fn
        self.name = name
        self.calls: List[str] = []

    def generate(self, prompt: str, **_) -> str:
        self.calls.append(prompt)
        if self.fn:
            return self.fn(prompt)
        if not self.replies:
            return ""
        return self.replies[(len(self.calls) - 1) % len(self.replies)]

In [ ]:
%%writefile ai_productivity_toolkit/templates.py
"""Reusable prompt templates (role, context, instruction, constraints, examples)."""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


_VAR = re.compile(r"\{(\w+)\}")


def _fill(text: str, values: Dict[str, object]) -> str:
    """Single-pass {var} substitution (safe for code / braces inside values)."""
    return _VAR.sub(lambda m: str(values.get(m.group(1), m.group(0))), text)


@dataclass
class PromptTemplate:
    name: str
    instruction: str
    role: str = ""
    context: str = ""
    constraints: List[str] = field(default_factory=list)
    examples: List[Tuple[str, str]] = field(default_factory=list)  # few-shot
    input_label: str = "Input"
    output_label: str = "Output"

    def with_examples(self, examples: List[Tuple[str, str]]) -> "PromptTemplate":
        """Return a copy turned into a few-shot template."""
        return PromptTemplate(self.name + "+fewshot", self.instruction, self.role,
                              self.context, list(self.constraints), list(examples),
                              self.input_label, self.output_label)

    def render(self, **values) -> str:
        parts: List[str] = []
        if self.role:
            parts.append(f"Role: {_fill(self.role, values)}")
        if self.context:
            parts.append(f"Context: {_fill(self.context, values)}")
        parts.append(f"Task: {_fill(self.instruction, values)}")
        if self.constraints:
            parts.append("Constraints:\n" + "\n".join(f"- {_fill(c, values)}" for c in self.constraints))
        for ex_in, ex_out in self.examples:
            parts.append(f"{self.input_label}: {ex_in}\n{self.output_label}: {ex_out}")
        parts.append(f"{self.input_label}: {values.get('input', '')}\n{self.output_label}:")
        return "\n\n".join(parts)

    @property
    def kind(self) -> str:
        return "few-shot" if self.examples else "zero-shot"

In [ ]:
%%writefile ai_productivity_toolkit/structured.py
"""Structured (JSON) output helpers for models that do not reliably emit JSON."""
from __future__ import annotations

import json
import re
from typing import Dict, Optional, Sequence


class StructuredOutputError(ValueError):
    pass


def extract_json(text: str) -> Optional[dict]:
    """Find the first valid JSON object in free-form model output."""
    cleaned = re.sub(r"```(?:json)?", "", text)
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", cleaned):
        try:
            obj, _ = decoder.raw_decode(cleaned[m.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None


def ask_json(llm, prompt: str, required_keys: Sequence[str], retries: int = 2) -> dict:
    """Ask for JSON, validate required keys, retry with a corrective prompt."""
    keys = ", ".join(required_keys)
    ask = f"{prompt}\n\nReturn ONLY a valid JSON object with exactly these keys: {keys}."
    last = ""
    for attempt in range(retries + 1):
        last = llm.generate(ask)
        data = extract_json(last)
        if data is not None and all(k in data for k in required_keys):
            return data
        ask = (f"{prompt}\n\nYour previous reply was not valid JSON with keys [{keys}]. "
               f"Reply with ONLY the JSON object, no extra text.")
    raise StructuredOutputError(f"No valid JSON after {retries + 1} tries. Last reply: {last!r}")


def extract_fields(llm, text: str, fields: Dict[str, str]) -> dict:
    """Reliable structured output for small models: ask one question per field,
    then assemble the JSON ourselves (so the result is always valid JSON)."""
    result = {}
    for key, question in fields.items():
        answer = llm.generate(f"{text}\n\nQuestion: {question}\nAnswer:")
        result[key] = answer.strip()
    return result

In [ ]:
%%writefile ai_productivity_toolkit/workflows.py
"""Ready-made productivity workflows built on prompt templates."""
from __future__ import annotations

import difflib
from typing import List, Optional, Sequence, Tuple

from .templates import PromptTemplate


SUMMARIZE = PromptTemplate(
    name="summarize",
    role="You are a careful editor.",
    instruction="Summarize the text below as {style}.",
    constraints=["Use at most {max_words} words.", "Do not add facts that are not in the text."],
    input_label="Text", output_label="Summary",
)

CLASSIFY = PromptTemplate(
    name="classify",
    instruction="Classify the text into exactly one label from: {labels}.",
    constraints=["Answer with the label only."],
    input_label="Text", output_label="Label",
)

EMAIL = PromptTemplate(
    name="email",
    role="You are a professional assistant who writes clear, polite emails.",
    context="The email is to {recipient}. Tone: {tone}.",
    instruction="Write a short email about: {purpose}.",
    constraints=["Include a subject line.", "Mention these points: {points}.", "Keep it under 120 words."],
    input_label="Request", output_label="Email",
)

RESUME = PromptTemplate(
    name="resume_bullet",
    role="You are a resume coach for the role of {role}.",
    instruction="Rewrite the resume bullet to be impactful: start with a strong action verb and show the result.",
    constraints=["One sentence.", "Do not invent numbers that are not given."],
    input_label="Original bullet", output_label="Improved bullet",
)

EXPLAIN_CODE = PromptTemplate(
    name="explain_code",
    role="You are a patient senior {language} developer.",
    instruction="Explain what the code does in simple words.",
    constraints=["Maximum 4 sentences."],
    input_label="Code", output_label="Explanation",
)

DEBUG_CODE = PromptTemplate(
    name="debug_code",
    role="You are a senior {language} developer doing code review.",
    instruction="Find the bug in the code and show the corrected line. Error message (if any): {error}.",
    constraints=["First state the bug in one sentence.", "Then give the fixed code."],
    input_label="Code", output_label="Review",
)

RESEARCH = PromptTemplate(
    name="research",
    role="You are a research assistant.",
    instruction="Create a short research outline for the topic with 4 key questions to investigate.",
    constraints=["Numbered list.", "Questions must be specific, not generic."],
    input_label="Topic", output_label="Outline",
)


def summarize(llm, text: str, style: str = "3 short bullet points", max_words: int = 60) -> str:
    return llm.generate(SUMMARIZE.render(input=text, style=style, max_words=max_words))


def classify(llm, text: str, labels: Sequence[str],
             examples: Optional[List[Tuple[str, str]]] = None) -> str:
    """Return one of `labels` (fuzzy-matched) or 'unknown'."""
    tpl = CLASSIFY.with_examples(examples) if examples else CLASSIFY
    raw = llm.generate(tpl.render(input=text, labels=", ".join(labels)))
    return normalize_label(raw, labels)


def normalize_label(raw: str, labels: Sequence[str]) -> str:
    low = raw.strip().lower()
    for lab in labels:
        if lab.lower() == low:
            return lab
    for lab in labels:
        if lab.lower() in low:
            return lab
    close = difflib.get_close_matches(low, [l.lower() for l in labels], n=1, cutoff=0.6)
    if close:
        return next(l for l in labels if l.lower() == close[0])
    return "unknown"


def write_email(llm, purpose: str, recipient: str = "a colleague", tone: str = "polite and professional",
                points: Sequence[str] = ()) -> str:
    prompt = EMAIL.render(input=purpose, purpose=purpose, recipient=recipient,
                          tone=tone, points="; ".join(points) or "none")
    return llm.generate(prompt, max_new_tokens=200)


def improve_resume_bullet(llm, bullet: str, role: str = "Software Engineer") -> str:
    return llm.generate(RESUME.render(input=bullet, role=role))


def explain_code(llm, code: str, language: str = "Python") -> str:
    return llm.generate(EXPLAIN_CODE.render(input=code, language=language))


def debug_code(llm, code: str, error: str = "none", language: str = "Python") -> str:
    return llm.generate(DEBUG_CODE.render(input=code, error=error, language=language))


def research_outline(llm, topic: str) -> str:
    return llm.generate(RESEARCH.render(input=topic))

In [ ]:
%%writefile ai_productivity_toolkit/evaluation.py
"""Prompt evaluation: test cases, check helpers, comparison and markdown reports."""
from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Callable, Dict, List, Sequence

from .structured import extract_json


Check = Callable[[str], bool]


def contains_any(*words: str) -> Check:
    return lambda out: any(w.lower() in out.lower() for w in words)


def max_words(n: int) -> Check:
    return lambda out: 0 < len(out.split()) <= n


def equals_label(label: str) -> Check:
    return lambda out: out.strip().lower() == label.lower()


def valid_json_with(*keys: str) -> Check:
    def _check(out: str) -> bool:
        data = extract_json(out)
        return data is not None and all(k in data for k in keys)
    return _check


@dataclass
class TestCase:
    name: str
    prompt: str
    check: Check
    check_desc: str = ""


@dataclass
class TestResult:
    name: str
    passed: bool
    seconds: float
    output: str
    check_desc: str


def run_suite(llm, cases: Sequence[TestCase]) -> List[TestResult]:
    results = []
    for c in cases:
        start = time.time()
        try:
            out = llm.generate(c.prompt)
        except Exception as e:  # keep the suite running
            out = f"[ERROR] {e}"
        results.append(TestResult(c.name, bool(c.check(out)), time.time() - start, out, c.check_desc))
    return results


def compare_prompts(llm, variants: Dict[str, str], check: Check) -> List[TestResult]:
    """Run the same task with several prompt wordings and score each."""
    return run_suite(llm, [TestCase(n, p, check) for n, p in variants.items()])


def to_markdown(results: Sequence[TestResult], backend_name: str = "unknown",
                title: str = "Prompt Testing Report") -> str:
    total = len(results)
    ok = sum(r.passed for r in results)
    lines = [f"# {title}", "",
             f"- Model/backend: `{backend_name}`",
             f"- Generated: {time.strftime('%Y-%m-%d %H:%M')}",
             f"- Result: **{ok}/{total} passed** ({(100 * ok // total) if total else 0}%)", "",
             "| # | Test | Check | Result | Time (s) |", "|---|------|-------|--------|----------|"]
    for i, r in enumerate(results, 1):
        lines.append(f"| {i} | {r.name} | {r.check_desc or '-'} | {'PASS' if r.passed else 'FAIL'} | {r.seconds:.2f} |")
    lines += ["", "## Outputs", ""]
    for r in results:
        out = r.output.replace("\n", " ").strip()
        lines.append(f"**{r.name}** ({'PASS' if r.passed else 'FAIL'}): {out[:300]}")
        lines.append("")
    return "\n".join(lines)

In [ ]:
%%writefile ai_productivity_toolkit/cli.py
"""Command line interface:  python -m ai_productivity_toolkit <command> [options]"""
from __future__ import annotations

import argparse
import sys
from typing import List, Optional

from . import __version__
from .backends import HFBackend, MockLLM
from .workflows import (
    classify, debug_code, explain_code, improve_resume_bullet,
    research_outline, summarize, write_email,
)


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="ai-toolkit", description="AI Productivity Toolkit")
    p.add_argument("--version", action="version", version=__version__)
    p.add_argument("--model", default="google/flan-t5-base", help="Hugging Face text2text model")
    p.add_argument("--mock", action="store_true", help="use a fake offline model (plumbing check)")
    sub = p.add_subparsers(dest="command", required=True)

    s = sub.add_parser("summarize", help="summarize text")
    s.add_argument("text")
    s.add_argument("--style", default="3 short bullet points")
    s.add_argument("--max-words", type=int, default=60)

    c = sub.add_parser("classify", help="classify text into one label")
    c.add_argument("text")
    c.add_argument("--labels", required=True, help="comma separated, e.g. positive,negative,neutral")

    e = sub.add_parser("email", help="draft an email")
    e.add_argument("purpose")
    e.add_argument("--to", default="a colleague")
    e.add_argument("--tone", default="polite and professional")
    e.add_argument("--point", action="append", default=[], help="point to mention (repeatable)")

    r = sub.add_parser("resume", help="improve a resume bullet")
    r.add_argument("bullet")
    r.add_argument("--role", default="Software Engineer")

    x = sub.add_parser("explain", help="explain code")
    x.add_argument("code")
    x.add_argument("--language", default="Python")

    d = sub.add_parser("debug", help="find the bug in code")
    d.add_argument("code")
    d.add_argument("--error", default="none")
    d.add_argument("--language", default="Python")

    q = sub.add_parser("research", help="create a research outline")
    q.add_argument("topic")
    return p


def main(argv: Optional[List[str]] = None) -> int:
    args = build_parser().parse_args(argv)
    llm = MockLLM(fn=lambda prompt: "[mock output]") if args.mock else HFBackend(args.model)

    if args.command == "summarize":
        out = summarize(llm, args.text, args.style, args.max_words)
    elif args.command == "classify":
        out = classify(llm, args.text, [l.strip() for l in args.labels.split(",") if l.strip()])
    elif args.command == "email":
        out = write_email(llm, args.purpose, args.to, args.tone, args.point)
    elif args.command == "resume":
        out = improve_resume_bullet(llm, args.bullet, args.role)
    elif args.command == "explain":
        out = explain_code(llm, args.code, args.language)
    elif args.command == "debug":
        out = debug_code(llm, args.code, args.error, args.language)
    else:
        out = research_outline(llm, args.topic)
    print(out)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile ai_productivity_toolkit/__main__.py
from .cli import main

raise SystemExit(main())

In [ ]:
import json
from ai_productivity_toolkit import *

llm = HFBackend("google/flan-t5-base")
print("Backend ready:", llm.name)

## 1. Zero-shot vs few-shot prompting
- **Zero-shot**: instruction only.
- **Few-shot**: instruction plus a few input/output examples that show the pattern.

In [ ]:
reviews = ["The food was cold and the waiter was rude.",
           "Absolutely loved the service, will come again!",
           "It was okay, nothing special."]
labels = ["positive", "negative", "neutral"]
examples = [("The phone broke after two days.", "negative"),
            ("Great quality and fast delivery.", "positive"),
            ("It works, but nothing special.", "neutral")]

print(f"{'Review':50} | zero-shot | few-shot")
for r in reviews:
    print(f"{r[:48]:50} | {classify(llm, r, labels):9} | {classify(llm, r, labels, examples=examples)}")

## 2. Roles, context, instructions and constraints
Same task and input, four prompt designs. Each is scored automatically (short answer?).

In [ ]:
text = ("Transformers use self-attention to weigh the importance of each token in a sentence. "
        "This lets language models understand context and generate fluent text. "
        "Training them needs a lot of data and computing power.")

variants = {
    "1 bare instruction":         f"Summarize: {text}",
    "2 + role":                   f"You are a teacher for beginners. Summarize: {text}",
    "3 + constraints":            f"Summarize the text in one sentence of at most 20 words. Text: {text}",
    "4 role+context+constraints": SUMMARIZE.render(input=text, style="one sentence", max_words=20),
}
for r in compare_prompts(llm, variants, max_words(25)):
    print("PASS" if r.passed else "FAIL", "|", r.name, "->", r.output)

## 3. Structured outputs (JSON)
1. `ask_json`: ask for JSON, validate keys, retry with a corrective prompt.
2. `extract_fields`: one question per field, JSON built by the toolkit (always valid; best for small models).

In [ ]:
note = "Hi, I am Ayesha Khan from Lahore. I want to book a meeting on Friday about the internship."

try:
    print("ask_json ->", ask_json(llm, f"Extract name, city and topic from: {note}", ["name", "city", "topic"], retries=1))
except StructuredOutputError as e:
    print("ask_json failed (common for small models):", str(e)[:120])

fields = {"name": "What is the person's name?", "city": "Which city is the person from?", "topic": "What is the meeting about?"}
print("extract_fields ->", json.dumps(extract_fields(llm, note, fields), indent=2))

## 4. Summarization, classification and generation

In [ ]:
article = ("Pakistan's IT exports have grown steadily as more freelancers and software houses sell services abroad. "
           "Experts say better internet and training programs can help young people earn in dollars. "
           "However, power cuts and payment issues still slow growth.")

print("SUMMARY:\n", summarize(llm, article, style="2 short bullet points", max_words=40))
print("\nCLASSIFY:", classify(llm, "My order arrived broken and support ignored me.", ["billing", "complaint", "praise", "question"]))
print("\nEMAIL:\n", write_email(llm, "ask for a deadline extension for my assignment", recipient="my teacher",
                                points=["I was sick", "need 2 extra days"]))

## 5. AI-assisted coding and debugging

In [ ]:
buggy = '''def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums) + 1
'''
print("EXPLAIN:\n", explain_code(llm, buggy))
print("\nDEBUG:\n", debug_code(llm, buggy, error="average([2, 4]) returns 4.0 but should be 3.0"))

## 6. Prompt evaluation and testing report
Each case has a prompt and an automatic check. The harness runs all cases and writes a markdown report.

In [ ]:
fewshot_cls = CLASSIFY.with_examples(examples)
cases = [
    TestCase("sentiment: negative review", fewshot_cls.render(input="Worst purchase ever.", labels="positive, negative, neutral"),
             lambda o: normalize_label(o, labels) == "negative", "label == negative"),
    TestCase("sentiment: positive review", fewshot_cls.render(input="Amazing, I love it!", labels="positive, negative, neutral"),
             lambda o: normalize_label(o, labels) == "positive", "label == positive"),
    TestCase("summary length", SUMMARIZE.render(input=text, style="one sentence", max_words=20), max_words(25), "<= 25 words"),
    TestCase("summary keeps key idea", SUMMARIZE.render(input=text, style="one sentence", max_words=25),
             contains_any("transformer", "attention", "context", "language"), "mentions key idea"),
    TestCase("extract city", f"{note}\n\nQuestion: Which city is the person from?\nAnswer:", contains_any("lahore"), "contains 'Lahore'"),
    TestCase("email looks like an email",
             EMAIL.render(input="deadline extension", purpose="deadline extension", recipient="my teacher",
                          tone="polite", points="need 2 extra days"),
             contains_any("subject", "dear", "extension"), "looks like an email"),
]
results = run_suite(llm, cases)
report = to_markdown(results, backend_name=llm.name, title="Prompt Testing Report")
print(report)
open("testing_report.md", "w").write(report)
print("\nSaved testing_report.md (download it from the file panel and put it in the repo's reports/ folder)")

## 7. The toolkit in action

In [ ]:
print("RESUME BULLET\n before: worked on website for a client\n after :",
      improve_resume_bullet(llm, "worked on website for a client", role="Web Developer"))
print("\nRESEARCH OUTLINE\n", research_outline(llm, "Impact of AI tools on student productivity"))

tweet = PromptTemplate("tweet", "Write a tweet about: {topic}", role="You are a social media writer.",
                       constraints=["Max 30 words", "Add 2 hashtags"])
print("\nCUSTOM TEMPLATE\n", llm.generate(tweet.render(input="", topic="learning prompt engineering")))

## 8. Key takeaways
- A prompt has parts: **role, context, instruction, constraints, examples**. Each part you add makes the output more controllable.
- **Few-shot** examples fix label and format confusion that zero-shot prompts leave open.
- Never trust a model to emit valid JSON: **validate and retry**, or build the JSON yourself field by field.
- Treat prompts like code: **test them** with automatic checks and keep a report.
- A small reusable toolkit with a single `generate()` seam lets you swap models without rewriting workflows.